In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from easydynamics.job import Job
from easydynamics.experiment import Experiment
from easydynamics.experiment import Data
from easydynamics.analysis import Analysis

from easydynamics.sample import BrownianTranslationalDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import Lorentzian
from easydynamics.sample import DeltaFunction
from easydynamics.sample import Polynomial

from easydynamics.sample import Gaussian

from easydynamics.resolution import ResolutionHandler

from easyscience import Parameter

import scipp as sc

import plopp as pp

%matplotlib widget

In [ ]:
# Create some fake data
Q=np.linspace(0.1,2,16)
E=np.linspace(-5,5,1001)

temperatures=[50,100,200,300]
diffusion_coefficients=[0.1,0.25,0.5,0.75]
convoluted_signal=np.zeros((len(temperatures),len(Q),len(E)))

scale=0.7 #arbitrary scale factor for diffusion model

for T in range(len(temperatures)):
    model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=diffusion_coefficients[T])
    HWHM=model.calculate_width(Q)

    QQISF=model.calculate_QISF(Q)
    EISF=model.calculate_EISF(Q)

    resolution=Gaussian(name="Resolution", area=1,width=0.1)

    resolution_handler=ResolutionHandler()

    sample_model=[]
    for i in range(len(Q)):
        sample_model.append(SampleModel(name=f"SampleModel_{i}"))

        sample_model[i].add_component(DeltaFunction(area=scale*EISF[i]+0.23, name="Elastic"))
        sample_model[i].add_component(Lorentzian(area=scale*QQISF[i], name="QuasiElastic", width=HWHM[i]) )

        convoluted_signal[T,i,:] = resolution_handler.convolve(E,sample_model[i],resolution)+0.45+0.1*np.random.normal(size=len(E))



Q_scipp=sc.array(dims=['Q'],values=Q, unit='1/angstrom')
E_scipp=sc.array(dims=['energy'],values=E,unit='meV')
intensity_scipp=sc.array(dims=['Temperature','Q','energy'],values=convoluted_signal,variances=0.1*convoluted_signal)

diffusion_data = sc.DataArray(data=intensity_scipp, coords={'Q':Q_scipp,'energy': E_scipp,'Temperature':sc.array(dims=['Temperature'],values=temperatures)})


pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['Q','energy'])



In [ ]:
pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['energy'])


In [ ]:


diffusion_job= Job(name='BrownianDiffusion')


exp=Experiment()
data=Data()
data.append(diffusion_data)

exp.set_data(data)

diffusion_job.set_experiment(exp)
diffusion_job.generate_empty_analysis_array()


bg=SampleModel('Background')
bg.add_component(Polynomial(coefficients=[0.5]))
diffusion_job.set_background_model(bg)
diffusion_job.set_background_model_for_all_analyses()

resolution=SampleModel()
resolution.add_component(Gaussian(name="Resolution", area=1,width=0.1))
diffusion_job.set_resolution_model(resolution)
diffusion_job.set_resolution_model_for_all_analyses()




diffusion_model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=0.3,scale=1.0)
diffusion_job.set_diffusion_model(diffusion_model)
# diffusion_job.set_theory_for_all_analyses(diffusion_model)



# delta_model=SampleModel(name="DeltaModel")
# delta_model.add_component(DeltaFunction(name="Delta",area=1.0))
# diffusion_job.set_theory_for_all_analyses(delta_model)

# delta_model=SampleModel(name="DeltaModel")
delta_model=DeltaFunction(name="Delta",area=0.2)
diffusion_job.set_theory_for_all_analyses(delta_model)


In [ ]:
diffusion_job._analysis

In [ ]:

# Quick check that the resolution model has been copied to all analyses with new parameters
print(diffusion_job.analysis[0][0]._resolution_model.components['Resolution'].area)
print(diffusion_job.analysis[0][0]._resolution_model.components['Resolution'].area.unique_name)
print(diffusion_job.analysis[1][0]._resolution_model.components['Resolution'].area.unique_name)
print(diffusion_job.analysis[1][0]._resolution_model.components['Resolution'].area)


In [ ]:
diffusion_job.analysis[2][5]._theory.components['Lorentzian'].width._independent

In [ ]:
diffusion_job.analysis[2][5].get_parameters()

In [ ]:
diffusion_job.analysis[2][5].get_fit_parameters()

In [ ]:
diffusion_job._diffusion_model.get_parameters()[0].fixed

In [ ]:
print(diffusion_job.analysis[0][0]._theory.components['Lorentzian'].width.dependency_expression)

print(diffusion_job.analysis[2][0]._theory.components['Lorentzian'].width.dependency_expression)

print(diffusion_job.analysis[2][5]._theory.components['Lorentzian'].width.dependency_expression)


In [ ]:
diffusion_job.analysis[2][5]._theory.components

In [ ]:
diffusion_job.plot_data_and_model(intensity_min=0.0, intensity_max=4,
                            energy_min=-5, energy_max=5)

In [ ]:
result=diffusion_job.fit_simultaneous()


In [ ]:
coords=diffusion_job._experiment._data.data.coords
coords

In [ ]:
diffusion_job._experiment._data.data.coords.get('Q').values

In [ ]:
diffusion_job._analysis_meta

In [ ]:
for key,variable in coords.items():
    print(key,variable)

In [ ]:
a=diffusion_job._analysis_meta
a['dims']

In [ ]:

a['sizes']['Temperature']

In [ ]:
iterator=np.ndindex(tuple(a['sizes'][dim] for dim in a['dims']))
for idx in iterator:
    print(idx)
    


In [ ]:
for ana in diffusion_job._analysis:
    print(ana)
    break

ana